Q2的任务如下：
如果运输系统运行不完美（例如系绳摇晃、火箭失效、电梯故障等），你的解决方案会在多大程度上改变？

这里需要考虑到系绳摇晃、火箭失效、电梯故障等等出错因素，对Q1中的方案的影响。

安装依赖包

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 配置matplotlib中文显示和负号显示，这里我就直接用英文了一劳永逸
plt.rcParams['font.family'] = 'DejaVu Sans'
# plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
# plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

# a. 仅使用太空电梯系统

考虑出错因素 系绳摇晃导致电梯运输速率下降、电梯故障导致部分电梯停运


In [ ]:
"""
多目标优化模型实现（考虑赖特定律）
"""

from scipy.optimize import minimize_scalar

# ==================== 参数配置 ====================

# 基础参数
total_materials = 100_000_000  # 总物资需求：1亿吨
total_materials_kg = total_materials * 1000

# 太空电梯参数
# 电梯基础参数
annual_capacity_per_harbor = 179_000  # 每个银河港口年运输能力：17.9万吨
num_harbors = 3  # 银河港口数量

# 电梯运输参数
harbor_annual_capacity = annual_capacity_per_harbor * num_harbors  # 每年运输吨数
harbor_cost_per_kg = 100  # 固定成本 100$/kg
C1_harbor = 100
b_harbor = 0.15  # 太空电梯学习率（较低，技术相对成熟）


# 电梯相关因素


# 火箭参数  
# 火箭基础参数
capacity_per_rocket = 150  # 每个火箭运输能力：100-150,这里取最大
num_launch_site = 10  # 火箭基地数量
num_launch_fre_per_day = 3 # 每个基地每天发送频率
rocket_cost_per_kg = 300 # 火箭运输成本300-500美刀/kg，这里取300

# 火箭运输参数
rocket_annual_capacity = 365 * num_launch_site * num_launch_fre_per_day * capacity_per_rocket  # 每年发射吨数
C1_rocket = 300  # 初始成本 300$/kg
b_rocket = 0.20  # 火箭学习率（较高，技术快速进步）

# 火箭相关因素
rocker_failrate = 0.05  # 火箭失败率5%



print("=" * 80)
print("多目标优化模型：时间-成本权衡分析".center(80))
print("=" * 80)


# ==================== 成本计算函数（考虑赖特定律） ====================

def calculate_cost_with_learning_annual_R(payload_kg, annual_capacity_tons, C1, b):
    """
    基于年份的赖特定律计算总成本
    
    参数:
        payload_kg: 需要运输的载荷 (kg)
        annual_capacity_tons: 年运输能力 (tons)
        C1: 初始单位成本 ($/kg)
        b: 学习率参数
    """
    if payload_kg == 0:
        return 0
    
    payload_tons = payload_kg / 1000
    years_needed = payload_tons / annual_capacity_tons
    
    # 逐年计算成本（考虑学习曲线）
    total_cost = 0
    cumulative_years = 0
    
    # 分年计算，每年应用当年的成本
    full_years = int(years_needed)
    for year in range(1, full_years + 1):
        unit_cost = wright_law_cost_annual_R(year, C1, b)
        yearly_payload_kg = annual_capacity_tons * 1000  # 转换为kg
        total_cost += unit_cost * yearly_payload_kg
    
    # 计算最后不足一年的部分
    remaining_years = years_needed - full_years
    if remaining_years > 0:
        unit_cost = wright_law_cost_annual_R(full_years + 1, C1, b)
        yearly_payload_kg = remaining_years * annual_capacity_tons * 1000
        total_cost += unit_cost * yearly_payload_kg
    
    avg_cost = total_cost / payload_kg
    return total_cost, avg_cost, years_needed

def calculate_cost_with_learning_annual_H(payload_kg, annual_capacity_tons, C1, b):
    """
    基于年份的赖特定律计算总成本
    
    参数:
        payload_kg: 需要运输的载荷 (kg)
        annual_capacity_tons: 年运输能力 (tons)
        C1: 初始单位成本 ($/kg)
        b: 学习率参数
    """
    if payload_kg == 0:
        return 0
    
    payload_tons = payload_kg / 1000
    years_needed = payload_tons / annual_capacity_tons
    
    # 逐年计算成本（考虑学习曲线）
    total_cost = 0
    cumulative_years = 0
    
    # 分年计算，每年应用当年的成本
    full_years = int(years_needed)
    for year in range(1, full_years + 1):
        unit_cost = wright_law_cost_annual_H(year, C1, b)
        yearly_payload_kg = annual_capacity_tons * 1000  # 转换为kg
        total_cost += unit_cost * yearly_payload_kg
    
    # 计算最后不足一年的部分
    remaining_years = years_needed - full_years
    if remaining_years > 0:
        unit_cost = wright_law_cost_annual_H(full_years + 1, C1, b)
        yearly_payload_kg = remaining_years * annual_capacity_tons * 1000
        total_cost += unit_cost * yearly_payload_kg
    
    avg_cost = total_cost / payload_kg
    return total_cost, avg_cost, years_needed


# ==================== 混合方案评估函数 ====================

def evaluate_hybrid_scheme(alpha, w1=0.5, w2=0.5, verbose=False):
    """
    评估混合方案的综合得分
    
    参数:
        alpha: 太空电梯运输比例 (0-1)
        w1: 时间权重
        w2: 成本权重
        verbose: 是否输出详细信息
    
    返回:
        objective: 目标函数值（越小越好）
        time_years: 所需时间（年）
        total_cost: 总成本（美元）
    """
    # 计算各自承担的运输量
    harbor_payload_kg = total_materials_kg * alpha
    rocket_payload_kg = total_materials_kg * (1 - alpha)
    
    # 计算太空电梯部分
    if alpha > 0:
        harbor_cost, harbor_avg_cost, harbor_years = calculate_cost_with_learning_annual_H(
            harbor_payload_kg, harbor_annual_capacity, C1_harbor, b_harbor
        )
    else:
        harbor_cost, harbor_avg_cost, harbor_years = 0, 0, 0
    
    # 计算火箭部分
    if alpha < 1:
        rocket_cost, rocket_avg_cost, rocket_years = calculate_cost_with_learning_annual_R(
            rocket_payload_kg, rocket_annual_capacity, C1_rocket, b_rocket
        )
    else:
        rocket_cost, rocket_avg_cost, rocket_years = 0, 0, 0
    
    # 总时间 = max(两者时间)，因为并行运输
    time_years = max(harbor_years, rocket_years)
    
    # 总成本 = 两者成本之和
    total_cost = harbor_cost + rocket_cost
    
    if verbose:
        print(f"\n方案分析 (α = {alpha:.3f}):")
        print(f"  太空电梯: {harbor_payload_kg/1e11:.2f}亿吨 -> {harbor_years:.2f}年, ${harbor_cost/1e9:.2f}亿")
        print(f"  火箭系统: {rocket_payload_kg/1e11:.2f}亿吨 -> {rocket_years:.2f}年, ${rocket_cost/1e9:.2f}亿")
        print(f"  总时间: {time_years:.2f}年, 总成本: ${total_cost/1e9:.2f}亿")
    
    return time_years, total_cost


# ==================== 计算基准值 ====================

print("\n计算基准值...")
print("-" * 80)

# 方案1: 仅太空电梯 (α=1)
time_harbor_only, cost_harbor_only = evaluate_hybrid_scheme(1.0, verbose=True)

# 方案2: 仅火箭 (α=0)
time_rocket_only, cost_rocket_only = evaluate_hybrid_scheme(0.0, verbose=True)

# 方案3: 均衡分配（最短时间）
# 为了使时间最短，需要让两者同时完成
# harbor_payload / harbor_capacity = rocket_payload / rocket_capacity
# alpha / harbor_capacity = (1-alpha) / rocket_capacity
# alpha * rocket_capacity = (1-alpha) * harbor_capacity
# alpha = harbor_capacity / (harbor_capacity + rocket_capacity)

alpha_balanced = harbor_annual_capacity / (harbor_annual_capacity + rocket_annual_capacity)
time_balanced, cost_balanced = evaluate_hybrid_scheme(alpha_balanced, verbose=True)

# 确定最小时间和最小成本
T_min = time_balanced
C_min = min(cost_harbor_only, cost_rocket_only)

print(f"\n基准值:")
print(f"  最短时间 T_min = {T_min:.2f} 年")
print(f"  最低成本 C_min = ${C_min/1e9:.2f} 亿美元")
print("-" * 80)


# b. 仅使用传统火箭


# c. 两者结合